# 27 AG retail rates

**Purpose.** To see what the mandate does to the rates that consumers pay, and not only to wholesale
payments, this notebook reads the ReEDS retail-rate module's outputs for each stage-1 run and for its
baseline `fbR_none`, and differences them (plan Phase 6; protocol amendment A21, draft).

The lab machine ran the module twice. The first runs used the module as it is in the repo
(`agent_comms/2026-11-05_retail_pilot/`, round 3, and `agent_comms/2026-11-05_retail_stage1/`). That module
leaves out capital rows whose technology name differs in case from `retail_eval_period`, among them all SMR
capital (O33, §6). Ethan chose a rerun from a patched copy (review card `retail-capex`); the lab ran it in
`agent_comms/2026-11-05_retail_patched/`. Sections 1–5 read the patched results. Section 6 reads the
unpatched results to show what the repo module leaves out, and checks the patched module against the same
rebuild. The module reads each run's output tables, which Claude dumped from the archived h5 files, and the
run's rebuilt `inputs_case`. This notebook opens no ReEDS outputs file. It reads the returned CSV files only.

| Input, per case (`results/<case>/` of a task folder) | Use |
|---|---|
| `retail_rate_components.csv` | revenue requirement by state, year and component (2004 $), and `retail_load` (MWh); patched for §1–5, both for §6 |
| `retail_rate_USA_centsperkWh.csv` | the module's national rate (2022 ¢/kWh), used only to check the rebuild in §2 |
| `agpatch_eval_unmatched.csv` (patched task) | the rows that the patched module still finds no period for, before its 20-year fill; §6 |
| `results/o30/<case>/` of the stage-1 task, and the supplied `outputs/<case>/` | the module's capital inputs, for §6 only |

**What it writes** (`results/exports/27/`): `rr_inputs.csv`, `rr_check_national.csv`, `rr_rates.csv`,
`rr_delta_components.csv`, `rr_delta_cendiv.csv`, `rr_consumer_gain.csv`, `rr_pv.csv`, `rr_capex_inputs.csv`,
`rr_capex_check.csv`, `rr_capex_patched_unmatched.csv`, `rr_capex_omitted.csv`, `rr_capex_omitted_dep.csv`,
`rr_o30_pumped_hydro.csv`. It states
no hypothesis.


In [1]:
import sys
from pathlib import Path
_p = Path.cwd().resolve()
while not (_p / "nlpaper").is_dir():           # find z-ethan/ from wherever the kernel starts
    _p = _p.parent
sys.path.insert(0, str(_p))
import hashlib

import numpy as np
import pandas as pd

import nlpaper
from nlpaper import paths
from aglib import ledger as L
from aglib import reduce as agr
nlpaper.setup(threads=4)
pd.set_option("display.width", 220)
pd.set_option("display.max_rows", 300)
pd.set_option("display.max_columns", 30)
OUT = paths.exports("27")
COMMS = paths.REPO / "z-ethan" / "agent_comms"
PATCHED = COMMS / "2026-11-05_retail_patched" / "results"
UNPATCHED = [COMMS / "2026-11-05_retail_pilot" / "results", COMMS / "2026-11-05_retail_stage1" / "results"]
MODULE = paths.REPO / "postprocessing" / "retail_rate_module"
TARGETS = ["eia", "aj", "iaea", "mck", "cop28", "eo"]
BASELINE = "fbR_none"
REF_RATE = "2.0%"
TO2025 = L.TO2025_FROM_2004

nlpaper setup: python 3.12.3 | numpy 1.26.4 | pandas 2.2.2 | matplotlib 3.8.4 | threads 4
determinism: seeds are explicit numpy Generators in each cell; results do not depend on PYTHONHASHSEED.


## 1. The returned files

A case counts when its folder holds both input files. `CASES` holds the patched results, which §1–5 use.
`CASES_U` holds the unpatched results: the pilot gave `smr100_eo_p50` and `fbR_none`, and the stage-1 task
the other five targets. Both sets must hold the same cases. Each file's MD5 is recorded, so a later rerun
can show which files it read.

In [2]:
FILES = ["retail_rate_components.csv", "retail_rate_USA_centsperkWh.csv"]
def find_cases(tasks):
    out = {}
    for task in tasks:
        for d in sorted(p for p in task.iterdir() if p.is_dir()):
            if all((d / f).is_file() for f in FILES):
                assert d.name not in out, f"{d.name} returned by two tasks"
                out[d.name] = d
    return out


CASES, CASES_U = find_cases([PATCHED]), find_cases(UNPATCHED)
assert BASELINE in CASES, "the baseline's retail outputs are missing"
assert sorted(CASES) == sorted(CASES_U), "patched and unpatched results hold different cases"
rows = []
for module, cs, files in (("patched", CASES, FILES + ["agpatch_eval_unmatched.csv"]), ("unpatched", CASES_U, FILES)):
    for c, d in cs.items():
        for f in files:
            b = (d / f).read_bytes()
            rows.append(dict(module=module, case=c, file=str((d / f).relative_to(paths.REPO)), md5=hashlib.md5(b).hexdigest(), bytes=len(b)))
INPUTS = pd.DataFrame(rows)
INPUTS.to_csv(OUT / "rr_inputs.csv", index=False)
PAIRS = [f"smr100_{t}_p50" for t in TARGETS if f"smr100_{t}_p50" in CASES]
print("cases:", list(CASES)); print("pairs against", BASELINE + ":", PAIRS)
display(INPUTS)

cases: ['fbR_none', 'smr100_aj_p50', 'smr100_cop28_p50', 'smr100_eia_p50', 'smr100_eo_p50', 'smr100_iaea_p50', 'smr100_mck_p50']
pairs against fbR_none: ['smr100_eia_p50', 'smr100_aj_p50', 'smr100_iaea_p50', 'smr100_mck_p50', 'smr100_cop28_p50', 'smr100_eo_p50']


,module,case,file,md5,bytes
0,patched,fbR_none,z-ethan\agent_comms\2026-11-05_retail_patched\...,bfe62b1ff39b777ad1e026614c5851a8,2815716
1,patched,fbR_none,z-ethan\agent_comms\2026-11-05_retail_patched\...,8e2b56b30b0a89e81edf5112855f5ef8,3346
2,patched,fbR_none,z-ethan\agent_comms\2026-11-05_retail_patched\...,7b50d7e351d6f7d196653e0b8041720e,11282
3,patched,smr100_aj_p50,z-ethan\agent_comms\2026-11-05_retail_patched\...,04d87c9c806c47dc6006c6c6f0ee9180,2815108
4,patched,smr100_aj_p50,z-ethan\agent_comms\2026-11-05_retail_patched\...,084d0809bc383ccee39570ee6af1a094,3347
5,patched,smr100_aj_p50,z-ethan\agent_comms\2026-11-05_retail_patched\...,b79019e536b146ee1190bf2b17b786dc,10127
6,patched,smr100_cop28_p50,z-ethan\agent_comms\2026-11-05_retail_patched\...,e8346648f6c7b48ef597f98d892eef2d,2813581
7,patched,smr100_cop28_p50,z-ethan\agent_comms\2026-11-05_retail_patched\...,1f3da43a568e7e227459a3fec6d85ce0,3347
8,patched,smr100_cop28_p50,z-ethan\agent_comms\2026-11-05_retail_patched\...,fd09a895cb5f0cefac697d0103c0568d,7849
9,patched,smr100_eia_p50,z-ethan\agent_comms\2026-11-05_retail_patched\...,bcac65571ef2a0351e29761857ebe827,2816466


## 2. Rebuild the module's national rate

The module's national rate (`retail_rate_calculations.get_dfplot`) is

rate(t) = [Σ_states Σ_components cost + bias] ÷ Σ_states `retail_load` × I ÷ 10,

with costs in 2004 $, load in MWh and I the module's deflator from 2004 to 2022 $ (the product of
`inputs/financials/inflation_default.csv` rates for 2005–2022, the file that `inflation_suffix = default`
selects). The bias term adds the state's mean bias correction (`inputs/state-meanbiaserror_rate-aggregation.csv`,
column named in `aggregation`, sign reversed) × `retail_load` × 10 ÷ I. The components are every column
except `state`, `t`, `busbar_load`, `end_use_load`, `distpv_gen` and `retail_load`.

The component file starts in 1960 and the module's national file in 2010; this notebook keeps 2010–2050.
Some cells are empty (before 2010 for the operating columns; `special_costs` in most rows). The module's
sums skip them, so they count as zero here too; the table below lists them.

The rebuild must equal the module's file in every year it writes.

In [3]:
infl = pd.read_csv(paths.REPO / "inputs" / "financials" / "inflation_default.csv", index_col="t")["inflation_rate"]
I_2022 = float(infl.loc[2005:2022].prod())
err = pd.read_csv(MODULE / "inputs" / "state-meanbiaserror_rate-aggregation.csv", index_col="index")
BIAS = -err.apply(lambda r: r[r["aggregation"]], axis=1)        # ¢/kWh in 2022 $
NONCOST = ["state", "t", "busbar_load", "end_use_load", "distpv_gen", "retail_load"]


NA = []
def components(case):
    d = pd.read_csv(CASES[case] / "retail_rate_components.csv", index_col=0)
    assert d["state"].isin(BIAS.index).all(), case
    na = d.isna()
    NA.extend(dict(case=case, column=c, empty_cells=int(na[c].sum()), empty_from_2010=int(na.loc[d["t"] >= 2010, c].sum()))
              for c in d.columns if na[c].any())
    d = d[d["t"] >= 2010].fillna(0.0)          # the module's sums skip empty cells, i.e. count them as zero
    d["bias_correction"] = d["state"].map(BIAS) * d["retail_load"] * 10 / I_2022
    return d


COMP = {c: components(c) for c in CASES}
COST = [x for x in COMP[BASELINE].columns if x not in NONCOST]
for c, d in COMP.items():
    assert [x for x in d.columns if x not in NONCOST] == COST, f"{c}: component columns differ from the baseline's"
    assert d.groupby("t").size().nunique() == 1 and d.groupby("t").size().iloc[0] == 48, c
EMPTY = pd.DataFrame(NA)
print("empty cells (all years, and from 2010); counted as zero, as the module's sums do:")
display(EMPTY.pivot_table(index="column", columns="case", values=["empty_cells", "empty_from_2010"]))
rows = []
for c, d in COMP.items():
    g = d.groupby("t")[COST + ["retail_load"]].sum()
    mine = g[COST].sum(axis=1) / g["retail_load"] * I_2022 / 10
    mod = pd.read_csv(CASES[c] / "retail_rate_USA_centsperkWh.csv").query("source == 'ReEDS'").set_index("t")["retailrate"]
    x = pd.concat([mine.rename("rebuilt"), mod.rename("module")], axis=1, join="inner")
    rows.append(x.assign(case=c, diff=x["rebuilt"] - x["module"]).reset_index())
CHK = pd.concat(rows, ignore_index=True)
CHK.to_csv(OUT / "rr_check_national.csv", index=False)
print(f"I(2004 -> 2022) = {I_2022:.6f}; {len(COST)} cost columns incl. bias; "
      f"{len(CHK)} case-years; largest |rebuilt - module| = {CHK['diff'].abs().max():.1e} 2022 ¢/kWh")
assert CHK["diff"].abs().max() < 1e-9

empty cells (all years, and from 2010); counted as zero, as the module's sums do:


empty_cells                                                                                            empty_from_2010                                                              \
case                             fbR_none smr100_aj_p50 smr100_cop28_p50 smr100_eia_p50 smr100_eo_p50 smr100_iaea_p50 smr100_mck_p50        fbR_none smr100_aj_p50 smr100_cop28_p50 smr100_eia_p50 smr100_eo_p50   
column                                                                                                                                                                                                             
op_acp_compliance_costs            2400.0        2400.0           2400.0         2400.0        2400.0          2400.0         2400.0             0.0           0.0              0.0            0.0           0.0   
op_co2_transport_storage           2400.0        2400.0           2400.0         2400.0        2400.0          2400.0         2400.0             0.0           0.0              0.0            0.0           0.0   
op_fuelcosts_objfn                 2400.0        2400.0           2400.0         2400.0        2400.0          2400.0         2400.0             0.0           0.0              0.0            0.0           0.0   
op_h2_transport_intrareg           2400.0        2400.0           2400.0         2400.0        2400.0          2400.0         2400.0             0.0           0.0              0.0            0.0           0.0   
op_h2combustion_fuel_costs         2400.0        2400.0           2400.0         2400.0        2400.0          2400.0         2400.0             0.0           0.0              0.0            0.0           0.0   
op_spurline_fom                    2400.0        2400.0           2400.0         2400.0        2400.0          2400.0         2400.0             0.0           0.0              0.0            0.0           0.0   
op_startcost                       2400.0        2400.0           2400.0         2400.0        2400.0          2400.0         2400.0             0.0           0.0              0.0            0.0           0.0   
op_transmission_intrazone_fom      2400.0        2400.0           2400.0         2400.0        2400.0          2400.0         2400.0             0.0           0.0              0.0            0.0           0.0   
op_vom_costs                       2400.0        2400.0           2400.0         2400.0        2400.0          2400.0         2400.0             0.0           0.0              0.0            0.0           0.0   
op_wc_debt_interest                1920.0        1920.0           1920.0         1920.0        1920.0          1920.0         1920.0             0.0           0.0              0.0            0.0           0.0   
op_wc_equity_return                1920.0        1920.0           1920.0         1920.0        1920.0          1920.0         1920.0             0.0           0.0              0.0            0.0           0.0   
op_wc_income_tax                   1920.0        1920.0           1920.0         1920.0        1920.0          1920.0         1920.0             0.0           0.0              0.0            0.0           0.0   
special_costs                      4352.0        4352.0           4352.0         4352.0        4352.0          4352.0         4352.0          1952.0        1952.0           1952.0         1952.0        1952.0   

                                                              
case                          smr100_iaea_p50 smr100_mck_p50  
column                                                        
op_acp_compliance_costs                   0.0            0.0  
op_co2_transport_storage                  0.0            0.0  
op_fuelcosts_objfn                        0.0            0.0  
op_h2_transport_intrareg                  0.0            0.0  
op_h2combustion_fuel_costs                0.0            0.0  
op_spurline_fom                           0.0            0.0  
op_startcost                              0.0            0.0  
op_tran

I(2004 -> 2022) = 1.546240; 56 cost columns incl. bias; 287 case-years; largest |rebuilt - module| = 1.2e-14 2022 ¢/kWh


## 3. Rates by case: national and census division

From here on, money is in 2025 $ through the ledger's deflator (`plotstyle.to2025(2004)`), not the module's.
A division's rate = Σ cost over its states ÷ Σ `retail_load` over its states (states → divisions:
`inputs/zones/state_groups.csv`). The module folds DC into MD. The national rate is the same ratio over
all 48 states, so it is the load-weighted mean of the division rates. Years are the module's annual years,
2010–2050; the module fills the years between solve years.

In [4]:
SCD = pd.read_csv(agr.STATE_CENDIV).set_index("st")["cendiv"]
rows = []
for c, d in COMP.items():
    assert d["state"].isin(SCD.index).all(), c
    x = d.assign(cost=d[COST].sum(axis=1), cendiv=d["state"].map(SCD))
    for scope, key in (("national", None), ("cendiv", "cendiv")):
        g = x.groupby(["t"] if key is None else [key, "t"])[["cost", "retail_load"]].sum().reset_index()
        g["region"] = "USA" if key is None else g.pop(key)
        rows.append(g.assign(case=c, scope=scope))
RATES = pd.concat(rows, ignore_index=True)
RATES["rate_c2025_per_kWh"] = RATES["cost"] / RATES["retail_load"] * TO2025 / 10
RATES["cost_2025B"] = RATES["cost"] * TO2025 / 1e9
RATES["retail_load_TWh"] = RATES["retail_load"] / 1e6
RATES = RATES[["case", "scope", "region", "t", "rate_c2025_per_kWh", "cost_2025B", "retail_load_TWh"]]
RATES.to_csv(OUT / "rr_rates.csv", index=False)
nat = RATES[RATES["scope"] == "national"].set_index(["t", "case"])
w = RATES[RATES["scope"] == "cendiv"].groupby(["t", "case"])[["cost_2025B", "retail_load_TWh"]].sum()
w = w.reindex(nat.index)
assert np.allclose(w["cost_2025B"], nat["cost_2025B"]) and np.allclose(w["retail_load_TWh"], nat["retail_load_TWh"]), "divisions sum to the nation"
display(nat["rate_c2025_per_kWh"].unstack("case").loc[[2026, 2030, 2035, 2040, 2045, 2050]].round(3))

case,fbR_none,smr100_aj_p50,smr100_cop28_p50,smr100_eia_p50,smr100_eo_p50,smr100_iaea_p50,smr100_mck_p50
t,,,,,,,
2026,14.629,14.630,14.630,14.630,14.630,14.630,14.630
2030,15.323,15.327,15.400,15.324,15.397,15.366,15.358
2035,16.013,16.036,16.316,16.014,16.347,16.183,16.172
2040,15.954,16.015,16.256,16.066,16.445,16.150,16.171
2045,15.703,15.797,16.149,15.772,16.443,15.881,15.971
2050,15.415,15.607,16.009,15.480,16.433,15.625,15.757


## 4. Differences: run − baseline

For each pair, Δrate = rate(run) − rate(baseline). The component rates sum to the rate in each case, so
Δrate splits exactly into Δ(component rate) = cost_g(run) ÷ load(run) − cost_g(base) ÷ load(base). Groups
follow the module's own plot categories (`retail_plots`): each column goes to the first category whose
pattern its name contains, in the module's order, so fixed O&M of spur lines and intrazone transmission
counts as fixed O&M. AC/DC converters and `op_trans` go to transmission, `special_costs` to
administration. "Flows between states and countries" are the module's payments for energy, planning
reserves and RPS credits traded across state lines and borders.

In [5]:
# retail_plots: the first category whose pattern the column name contains, in this order
CATS = [("_gen_", "generation capital"), ("_admin", "administration"), ("_dist", "distribution"), ("_fuel", "fuel"),
        ("_fom", "fixed O&M"), ("_flow", "flows between states and countries"), ("_spurline", "spur lines"),
        ("_trans_FERC", "transmission"), ("_transmission", "transmission"), ("op_", "other operating"),
        ("bias_correction", "bias correction")]
CREDITS = ["ptc_grossup", "itc_normalized_value", "itc_trans"]


def group(col):
    # post_processing first: special costs into administration, converters into transmission
    if col == "special_costs":
        return "administration"
    if col.startswith("cap_converter_") or col == "op_trans":      # retail_plots moves op_trans to transmission
        return "transmission"
    for pat, lab in CATS:
        if pat in col:
            return lab
    assert col in CREDITS, col
    return "tax credits"


GROUP = pd.Series({c: group(c) for c in COST})
display(GROUP.index.to_series().groupby(GROUP.values).apply(lambda x: ", ".join(x)).to_frame("columns"))


def comp_rates(case):
    g = COMP[case].groupby("t")[COST + ["retail_load"]].sum()
    return g[COST].div(g["retail_load"], axis=0).T.groupby(GROUP).sum().T * TO2025 / 10   # 2025 ¢/kWh


rows = []
base = comp_rates(BASELINE)
for c in PAIRS:
    d = comp_rates(c) - base
    rows.append(d.stack().rename("delta_c2025_per_kWh").reset_index().rename(columns={"level_1": "group"})
                .assign(case=c, target=c.split("_")[1]))
DCOMP = pd.concat(rows, ignore_index=True)
DCOMP.to_csv(OUT / "rr_delta_components.csv", index=False)
tot = DCOMP.groupby(["case", "t"])["delta_c2025_per_kWh"].sum().unstack("case")
chk = nat["rate_c2025_per_kWh"].unstack("case")
assert np.allclose(tot, chk.loc[tot.index, tot.columns].sub(chk.loc[tot.index, BASELINE], axis=0)), "components sum to Δrate"
print("national Δrate, 2025 ¢/kWh:")
display(tot.loc[[2026, 2030, 2035, 2040, 2045, 2050]].round(3))
print("components of national Δrate in 2050, 2025 ¢/kWh:")
display(DCOMP[DCOMP["t"] == 2050].pivot_table(index="group", columns="target", values="delta_c2025_per_kWh").round(3))

,columns
administration,"special_costs, op_admin, cap_admin_dep_expense..."
bias correction,bias_correction
distribution,"op_dist, cap_dist_dep_expense, cap_dist_debt_i..."
fixed O&M,"op_fom_costs, op_spurline_fom, op_transmission..."
flows between states and countries,"load_flow, rps_flow, res_marg_ann_flow"
fuel,"op_fuelcosts_objfn, op_h2combustion_fuel_costs"
generation capital,"cap_gen_dep_expense, cap_gen_debt_interest, ca..."
other operating,"op_acp_compliance_costs, op_co2_transport_stor..."
spur lines,"cap_spurline_dep_expense, cap_spurline_debt_in..."
tax credits,"ptc_grossup, itc_normalized_value, itc_trans"


national Δrate, 2025 ¢/kWh:


case,smr100_aj_p50,smr100_cop28_p50,smr100_eia_p50,smr100_eo_p50,smr100_iaea_p50,smr100_mck_p50
t,,,,,,
2026,0.000,0.001,0.000,0.000,0.001,0.001
2030,0.004,0.077,0.001,0.073,0.043,0.035
2035,0.023,0.303,0.001,0.333,0.170,0.159
2040,0.061,0.302,0.112,0.490,0.196,0.217
2045,0.095,0.446,0.069,0.740,0.179,0.269
2050,0.192,0.594,0.065,1.018,0.210,0.342


components of national Δrate in 2050, 2025 ¢/kWh:


target,aj,cop28,eia,eo,iaea,mck
group,,,,,,
administration,0.000,0.000,0.000,0.000,0.000,0.000
bias correction,0.000,0.000,-0.000,0.000,0.000,0.000
distribution,0.000,0.000,0.000,0.000,0.000,0.000
fixed O&M,0.037,0.366,0.030,0.367,0.150,0.150
flows between states and countries,-0.001,-0.006,-0.000,-0.008,-0.002,-0.003
fuel,-0.032,-0.253,-0.026,-0.396,-0.063,-0.099
generation capital,0.246,0.644,0.087,1.254,0.222,0.413
other operating,-0.004,0.007,-0.002,0.004,0.002,0.002
spur lines,-0.024,-0.086,-0.009,-0.105,-0.042,-0.056


**Division Δrate.** The same difference by census division, with each division's retail load in both
runs.

In [6]:
cd = RATES[RATES["scope"] == "cendiv"].set_index(["region", "t", "case"])
rows = []
for c in PAIRS:
    r, b = cd.xs(c, level="case"), cd.xs(BASELINE, level="case")
    rows.append(pd.DataFrame({"delta_c2025_per_kWh": r["rate_c2025_per_kWh"] - b["rate_c2025_per_kWh"],
                              "retail_load_TWh_run": r["retail_load_TWh"], "retail_load_TWh_base": b["retail_load_TWh"]})
                .reset_index().assign(case=c, target=c.split("_")[1]))
DCD = pd.concat(rows, ignore_index=True).rename(columns={"region": "cendiv"})
DCD.to_csv(OUT / "rr_delta_cendiv.csv", index=False)
for c in PAIRS:
    print(c)
    display(DCD[DCD["case"] == c].pivot_table(index="cendiv", columns="t", values="delta_c2025_per_kWh")
            [[2030, 2035, 2040, 2045, 2050]].round(3))

smr100_eia_p50


t,2030,2035,2040,2045,2050
cendiv,,,,,
East_North_Central,0.000,0.000,0.038,0.060,0.025
East_South_Central,0.000,0.000,-0.007,-0.019,-0.016
Mid_Atlantic,0.000,0.000,0.361,0.294,0.302
Mountain,0.008,0.011,0.396,0.276,0.350
New_England,0.000,0.000,-0.103,-0.218,-0.068
Pacific,0.000,0.000,-0.015,-0.027,-0.062
South_Atlantic,0.000,0.000,0.117,0.040,0.034
West_North_Central,0.000,0.000,0.066,0.048,0.008
West_South_Central,0.000,0.000,0.088,0.058,0.041


smr100_aj_p50


t,2030,2035,2040,2045,2050
cendiv,,,,,
East_North_Central,0.001,0.054,0.013,0.014,0.107
East_South_Central,-0.000,-0.028,0.000,-0.016,-0.042
Mid_Atlantic,0.030,0.084,0.253,0.474,0.863
Mountain,0.023,0.146,0.382,0.445,0.725
New_England,-0.004,-0.026,-0.028,-0.140,0.028
Pacific,-0.002,-0.012,-0.038,-0.025,-0.035
South_Atlantic,-0.001,-0.003,-0.006,0.018,0.116
West_North_Central,-0.005,-0.024,0.013,0.145,0.201
West_South_Central,-0.001,0.008,0.048,0.029,0.021


smr100_iaea_p50


t,2030,2035,2040,2045,2050
cendiv,,,,,
East_North_Central,0.044,0.450,0.457,0.400,0.398
East_South_Central,-0.001,-0.064,-0.037,-0.061,-0.093
Mid_Atlantic,0.034,0.145,0.337,0.553,0.724
Mountain,0.205,0.329,0.418,0.467,0.627
New_England,-0.005,-0.027,-0.242,-0.463,-0.241
Pacific,-0.025,0.044,0.040,-0.053,-0.136
South_Atlantic,0.096,0.217,0.186,0.146,0.173
West_North_Central,-0.015,0.212,0.279,0.313,0.400
West_South_Central,-0.002,0.022,0.058,0.022,0.004


... (2 outputs left out)

smr100_cop28_p50


t,2030,2035,2040,2045,2050
cendiv,,,,,
East_North_Central,0.060,0.733,0.684,1.123,1.485
East_South_Central,-0.006,-0.177,-0.049,-0.085,0.101
Mid_Atlantic,0.073,0.152,0.311,0.539,0.809
Mountain,0.244,0.533,0.742,0.881,1.248
New_England,-0.026,-0.032,-0.234,-0.476,-0.388
Pacific,-0.041,0.063,0.024,-0.076,-0.236
South_Atlantic,0.218,0.469,0.330,0.477,0.626
West_North_Central,0.011,0.438,0.446,0.596,0.664
West_South_Central,-0.004,0.082,0.107,0.289,0.350


smr100_eo_p50


t,2030,2035,2040,2045,2050
cendiv,,,,,
East_North_Central,0.082,0.773,1.005,1.668,2.206
East_South_Central,-0.003,-0.052,0.155,0.284,0.462
Mid_Atlantic,0.063,0.106,0.322,0.582,0.862
Mountain,0.223,0.613,1.091,1.309,1.831
New_England,-0.014,-0.032,-0.257,-0.472,-0.339
Pacific,-0.053,0.078,-0.014,-0.051,-0.263
South_Atlantic,0.205,0.505,0.518,0.682,1.198
West_North_Central,0.010,0.439,0.583,0.771,0.742
West_South_Central,-0.006,0.102,0.403,0.808,1.021


## 5. Consumers' gain: retail beside wholesale

Consumers' retail bills equal the module's revenue requirement, Σ cost, so the consumers' retail gain is
−Δ Σ cost (2025 $B per year, + = gain). Beside it stands notebook 21's gain on wholesale market payments
(energy + capacity + RPS; `ag_annual_A.csv`, horizon A, reference keys: lump support, funding T,
allowance keys free or none). The two measure different things: the retail figure includes the
regulated recovery of capital, transmission, distribution and administration, and the module's own
financing assumptions. No reconciliation is attempted here.

PV: the retail gain at each ledger rate, 2026–2050, discounted to 2026, 2025 $B. The module's revenue
requirement is already an annual flow, so it is counted year by year, as horizon A counts annual flows.

In [7]:
A21 = pd.read_csv(paths.exports("21") / "ag_annual_A.csv")
A21 = A21[(A21["actor"] == "electricity consumers") & A21["entry"].isin(["energy payments", "capacity payments", "RPS payments"])
          & (A21["support_form"] == "lump") & (A21["funding"] == "T") & A21["allowance_national"].isin(["free", "none"])
          & A21["allowance_state"].isin(["free", "none"])]
assert A21.groupby(["target", "entry", "year"]).size().max() == 1
whole = A21.groupby(["target", "year"])["gain_2025B"].sum()
rows = []
for c in PAIRS:
    t = c.split("_")[1]
    g = -(nat.xs(c, level="case")["cost_2025B"] - nat.xs(BASELINE, level="case")["cost_2025B"])
    g = g.loc[2026:2050]
    rows.append(pd.DataFrame({"retail_gain_2025B": g, "wholesale_gain_2025B": whole.loc[t].reindex(g.index)})
                .rename_axis("t").reset_index().assign(case=c, target=t))
GAIN = pd.concat(rows, ignore_index=True)
GAIN.to_csv(OUT / "rr_consumer_gain.csv", index=False)
pv = []
for c, g in GAIN.groupby("case"):
    for rk, r in L.RATES.items():
        f = (1.0 + r) ** -(g["t"].values - 2026)
        pv.append(dict(case=c, target=g["target"].iloc[0], rate=rk, retail_gain_pv_2025B=float(g["retail_gain_2025B"] @ f),
                       wholesale_gain_pv_2025B=float(g["wholesale_gain_2025B"].fillna(0.0) @ f)))
PV = pd.DataFrame(pv)
PV.to_csv(OUT / "rr_pv.csv", index=False)
display(GAIN[GAIN["t"].isin([2026, 2030, 2035, 2040, 2045, 2050])].pivot_table(
    index="t", columns="target", values=["retail_gain_2025B", "wholesale_gain_2025B"]).round(2))
display(PV[PV["rate"] == REF_RATE].round(1))

retail_gain_2025B                                   wholesale_gain_2025B                                  
target                aj  cop28   eia     eo   iaea    mck                   aj  cop28   eia     eo   iaea    mck
t                                                                                                                
2026               -0.02  -0.03 -0.02  -0.02  -0.03  -0.02                 0.00   0.00  0.00   0.00   0.00   0.00
2030               -0.26  -3.55 -0.03  -3.37  -2.01  -1.61                 0.15   2.12  0.00   2.94   0.69   0.90
2035               -1.30 -15.48 -0.05 -16.96  -8.78  -8.14                 0.05   9.86  0.00  16.40   3.86   3.77
2040               -3.53 -16.83 -6.19 -27.20 -11.01 -12.11                 3.28  15.66  4.69  33.01   9.19   9.57
2045               -5.71 -26.60 -4.14 -44.10 -10.71 -16.03                 4.63  37.81  2.71  62.84  12.94  16.17
2050              -12.17 -37.52 -4.09 -64.30 -13.36 -21.65                13.12  61.07  2.51  80.80  18.17  27.28

,case,target,rate,retail_gain_pv_2025B,wholesale_gain_pv_2025B
1,smr100_aj_p50,aj,2.0%,-60.1,51.1
7,smr100_cop28_p50,cop28,2.0%,-298.3,353.4
13,smr100_eia_p50,eia,2.0%,-48.7,32.7
19,smr100_eo_p50,eo,2.0%,-453.0,585.9
25,smr100_iaea_p50,iaea,2.0%,-147.6,133.4
31,smr100_mck_p50,mck,2.0%,-180.2,162.6


## 6. Capital rows the module leaves out (O30, O33)

The module turns each generator's capital spending into depreciation, return on rate base and income tax
over an evaluation period (`retail_rate_calculations.py:1501-1595`). It looks the period up in
`inputs_case/retail_eval_period` by technology name (new builds: lines 878-889, on name and year; historical
builds: lines 907-930, on name at the first year). The lookup matches names exactly. The run's output tables
and `df_capex_init.csv` use lower-case names (`nuclear-smr`, `gas-cc`), while `retail_eval_period` mixes case
(`Nuclear-SMR`, `Gas-CC`). A row with no match gets an empty period. The module then fills empty periods
with 20 years through `df[col].fillna(20, inplace=True)`. Under pandas 3 that call changes nothing: it acts
on a copy and raises `ChainedAssignmentError`, which the lab logged at lines 883, 889, 926 and 930
(`retail_stage1/response.md`, F5). A row with an empty period matches no plant age, so it has no year and
drops out of every sum.

**Check 1 (unpatched module).** If that reading is right, the unpatched module's national
`cap_gen_dep_expense` must equal the depreciation of the matched rows alone. This section rebuilds it from the lab's extract (`o30_df_capex_init.csv`,
`o30_retail_eval_period.csv`) and the supplied output tables: `capex_ivrt.csv` and the investment entries of
`systemcost_techba.csv`, taking the larger of the two and spreading it between solve years as the module does
(lines 772-803, `distribute_between_solve_years`). Depreciation is straight-line, with the module's inflation
adjustment (`inputs/financials/inflation_default.csv`). Solve years come from `load_cat.csv`.

**Check 2 (patched module).** The patched copy matches names ignoring case and assigns the 20-year fill
(`retail_patched/reference/patch_expected.diff`). Its national `cap_gen_dep_expense` must therefore equal the
same rebuild over all rows, with each row's period read as (b) below. The rows it still finds no period for
(`agpatch_eval_unmatched.csv`) must equal the rows that reading (b) leaves without a name.

**Size.** For the rows that drop out, the depreciation they would add under two readings: (a) the module's
stated default of 20 years for every unmatched row; (b) the period of the same name ignoring case, else 20
years. Depreciation is one of the four capital lines, so neither reading gives the full capital recovery of
these rows; return on rate base and income tax come on top. Neither reading is a proposed fix.

In [8]:
CAPDIR = COMMS / "2026-11-05_retail_stage1" / "results" / "o30"
def outdir(case):
    for t in ["2026-11-05_retail_pilot", "2026-11-05_retail_stage1"]:
        d = COMMS / t / "outputs" / case
        if d.is_dir():
            return d
    raise FileNotFoundError(case)


CAP_FILES = [CAPDIR / c / f for c in CASES for f in ["o30_df_capex_init.csv", "o30_retail_eval_period.csv", "o30_rsc_dat.csv"]]
CAP_FILES += [outdir(c) / f for c in CASES for f in ["capex_ivrt.csv", "systemcost_techba.csv", "load_cat.csv"]]
CAPIN = pd.DataFrame([dict(file=str(f.relative_to(paths.REPO)), md5=hashlib.md5(f.read_bytes()).hexdigest()) for f in CAP_FILES])
CAPIN.to_csv(OUT / "rr_capex_inputs.csv", index=False)
INV = ["inv_dac", "inv_h2_production", "inv_investment_capacity_costs", "inv_investment_refurbishment_capacity",
       "inv_investment_spurline_costs_rsc_technologies"]                     # retail_rate_calculations.py:784-786


def dep_by_year(rows, y0, y1):
    # straight-line depreciation with the module's inflation adjustment (lines 1501-1535, 1564-1565)
    out = {}
    for r in rows.itertuples():
        for age in range(int(r.eval_period)):
            y = r.t + age
            adj = 1.0 if age == 0 else 1.0 / float(infl.loc[r.t + 1:y].prod())
            out[y] = out.get(y, 0.0) + r.capex * adj / r.eval_period
    return pd.Series(out, dtype=float).reindex(range(y0, y1 + 1)).fillna(0.0)


def capital_rows(case):
    ev = pd.read_csv(CAPDIR / case / "o30_retail_eval_period.csv")
    od = outdir(case)
    modeled = sorted(pd.read_csv(od / "load_cat.csv")["t"].unique())
    first, last = modeled[0], modeled[-1]
    yx = [(first, first, 1.0)]
    for y in range(first + 1, last + 1):
        pre, fol = max(m for m in modeled if m < y), min(m for m in modeled if m >= y)
        yx.append((y, fol, 1.0 / (fol - pre)))
    yx = pd.DataFrame(yx, columns=["t", "t_modeled", "alloc_f"])
    cx = pd.read_csv(od / "capex_ivrt.csv").groupby(["i", "r", "t"])["Value"].sum()
    sc = pd.read_csv(od / "systemcost_techba.csv")
    sc = sc[sc["sys_costs"].isin(INV)].groupby(["i", "r", "t"])["Value"].sum()
    new = pd.concat([cx, sc], axis=1).max(axis=1).rename("capex").reset_index().rename(columns={"t": "t_modeled"})
    new = new.merge(yx, on="t_modeled", how="left")
    new["capex"] = new["capex"] * new["alloc_f"]
    new = new.groupby(["i", "t"], as_index=False)["capex"].sum().merge(ev, on=["i", "t"], how="left").assign(src="new")
    comp = pd.read_csv(CASES_U[case] / "retail_rate_components.csv", index_col=0)
    hist = pd.read_csv(CAPDIR / case / "o30_df_capex_init.csv")
    hist = hist[(hist["t"] <= first) & (hist["t"] >= comp["t"].min())].groupby(["i", "t"], as_index=False)["capex"].sum()
    hist = hist.merge(ev[ev["t"] == first][["i", "eval_period"]], on="i", how="left").assign(src="historical")
    evl = ev.assign(il=ev["i"].str.lower()).drop_duplicates(["il", "t"])
    new["eval_ci"] = new.assign(il=new["i"].str.lower()).merge(evl[["il", "t", "eval_period"]], on=["il", "t"], how="left", suffixes=("_x", ""))["eval_period"].values
    hist["eval_ci"] = hist.assign(il=hist["i"].str.lower()).merge(evl[evl["t"] == first][["il", "eval_period"]], on="il", how="left", suffixes=("_x", ""))["eval_period"].values
    return pd.concat([new, hist], ignore_index=True), comp, int(comp["t"].min()), last


ROWS, CHECK, OMIT, DEP, UNM = {}, [], [], [], []
for c in CASES:
    rows, comp, y0, y1 = capital_rows(c)
    ROWS[c] = rows
    module = comp.groupby("t")["cap_gen_dep_expense"].sum().reindex(range(y0, y1 + 1)).fillna(0.0)
    kept = dep_by_year(rows.dropna(subset=["eval_period"]), y0, y1)
    CHECK.append(dict(case=c, module="unpatched", rebuild="matched rows only", years=f"{y0}-{y1}",
                      max_abs_err=float((kept - module).abs().max()), max_abs_module=float(module.abs().max())))
    comp_p = pd.read_csv(CASES[c] / "retail_rate_components.csv", index_col=0)
    module_p = comp_p.groupby("t")["cap_gen_dep_expense"].sum().reindex(range(y0, y1 + 1)).fillna(0.0)
    full = dep_by_year(rows.assign(eval_period=rows["eval_ci"].fillna(20.0)), y0, y1)
    CHECK.append(dict(case=c, module="patched", rebuild="all rows, name ignoring case, else 20", years=f"{y0}-{y1}",
                      max_abs_err=float((full - module_p).abs().max()), max_abs_module=float(module_p.abs().max())))
    um = pd.read_csv(CASES[c] / "agpatch_eval_unmatched.csv")
    um = um[um["t"] >= y0].groupby(["src", "i"])["capex"].sum()
    mine = rows[rows["eval_ci"].isna()].groupby(["src", "i"])["capex"].sum()
    UNM.append(pd.concat([um.rename("module_capex"), mine.rename("rebuild_capex")], axis=1).reset_index().assign(case=c))
    miss = rows[rows["eval_period"].isna()]
    OMIT.append(miss.groupby(["src", "i"], as_index=False)["capex"].sum().assign(case=c))
    load = comp.groupby("t")["retail_load"].sum().reindex(range(2026, y1 + 1))
    for reading, col in [("default 20 years", None), ("name ignoring case, else 20", "eval_ci")]:
        m = miss.assign(eval_period=20.0 if col is None else miss[col].fillna(20.0))
        d = dep_by_year(m, 2026, y1)
        DEP.append(pd.DataFrame({"t": d.index, "case": c, "reading": reading, "omitted_dep_2004B": d.values / 1e9,
                                 "omitted_dep_c2025_per_kWh": (d / load / 10 * TO2025).values}))
CHECK = pd.DataFrame(CHECK)
CHECK.to_csv(OUT / "rr_capex_check.csv", index=False)
display(CHECK)
assert (CHECK["max_abs_err"] < 1e-2).all(), "the rebuilds reproduce each module's cap_gen_dep_expense"
UNM = pd.concat(UNM, ignore_index=True)
UNM.to_csv(OUT / "rr_capex_patched_unmatched.csv", index=False)
print("rows the patched module still finds no period for (before its 20-year fill), 2004 $B:")
display(UNM.assign(module_capex=UNM["module_capex"] / 1e9, rebuild_capex=UNM["rebuild_capex"] / 1e9)
        .pivot_table(index=["src", "i"], columns="case", values="module_capex").round(2))
assert np.allclose(UNM["module_capex"], UNM["rebuild_capex"], rtol=1e-9), "patched module's unmatched rows = rebuild's"
OMIT = pd.concat(OMIT, ignore_index=True).assign(capex_2004B=lambda x: x["capex"] / 1e9).drop(columns="capex")
OMIT.to_csv(OUT / "rr_capex_omitted.csv", index=False)
tot = OMIT.pivot_table(index="case", columns="src", values="capex_2004B", aggfunc="sum")
allc = pd.DataFrame({c: ROWS[c].groupby("src")["capex"].sum() / 1e9 for c in CASES}).T
print("capital spending with no exact eval_period name, and all capital spending, 2004 $B:")
display(pd.concat([tot.add_prefix("omitted "), allc.add_prefix("all ")], axis=1).round(1))
print("omitted, by technology, 2004 $B:")
display(OMIT.pivot_table(index=["src", "i"], columns="case", values="capex_2004B").round(1))

,case,module,rebuild,years,max_abs_err,max_abs_module
0,fbR_none,unpatched,matched rows only,1960-2050,0.000053,3.868110e+10
1,fbR_none,patched,"all rows, name ignoring case, else 20",1960-2050,0.000092,5.692311e+10
2,smr100_aj_p50,unpatched,matched rows only,1960-2050,0.000053,3.864755e+10
3,smr100_aj_p50,patched,"all rows, name ignoring case, else 20",1960-2050,0.000099,5.719291e+10
4,smr100_cop28_p50,unpatched,matched rows only,1960-2050,0.000053,3.731357e+10
5,smr100_cop28_p50,patched,"all rows, name ignoring case, else 20",1960-2050,0.000092,6.045395e+10
6,smr100_eia_p50,unpatched,matched rows only,1960-2050,0.000053,3.868110e+10
7,smr100_eia_p50,patched,"all rows, name ignoring case, else 20",1960-2050,0.000076,5.712915e+10
8,smr100_eo_p50,unpatched,matched rows only,1960-2050,0.000065,3.688774e+10
9,smr100_eo_p50,patched,"all rows, name ignoring case, else 20",1960-2050,0.000114,6.696400e+10


rows the patched module still finds no period for (before its 20-year fill), 2004 $B:


case                         fbR_none  smr100_aj_p50  smr100_cop28_p50  smr100_eia_p50  smr100_eo_p50  smr100_iaea_p50  smr100_mck_p50
src i                                                                                                                                 
new CoalOldScr_coal-CCS_mod     70.81          70.78             70.01           70.81          68.72            70.85           70.86
    CoalOldUns_coal-CCS_mod      5.54           5.55              5.35            5.54           5.06             5.54            5.48
    Gas-CC_Gas-CC-CCS_mod        6.63           3.99               NaN            3.79           0.14             1.41            1.29
    Gas-CC_H2-CC                 0.01           0.01              0.01            0.01           0.01             0.00            0.01
    coal-new_coal-CCS_mod        6.46           6.46              4.91            6.46           4.91             5.33            4.91

capital spending with no exact eval_period name, and all capital spending, 2004 $B:


src,omitted historical,omitted new,all historical,all new
fbR_none,1405.5,779.6,1644.0,2631.9
smr100_aj_p50,1355.3,960.4,1593.7,2747.5
smr100_cop28_p50,1332.2,1398.1,1570.7,2995.4
smr100_eia_p50,1405.5,878.4,1644.0,2695.6
smr100_eo_p50,1379.8,1829.6,1618.2,3341.9
smr100_iaea_p50,1332.2,1050.1,1570.7,2772.0
smr100_mck_p50,1405.5,1173.9,1644.0,2865.8


omitted, by technology, 2004 $B:


case                                fbR_none  smr100_aj_p50  smr100_cop28_p50  smr100_eia_p50  smr100_eo_p50  smr100_iaea_p50  smr100_mck_p50
src        i                                                                                                                                 
historical coaloldscr                  289.3          289.3             289.3           289.3          289.3            289.3           289.3
           coalolduns                   33.1           33.1              33.1            33.1           33.1             33.1            33.1
           gas-cc                      199.8          199.8             199.8           199.8          199.8            199.8           199.8
           gas-ct                       92.8           92.8              92.8            92.8           92.8             92.8            92.8
           hydnd                         4.0            4.0               4.0             4.0            4.0              4.0             4.0
           nuclear                     786.5          736.3             713.2           786.5          760.7            713.2           786.5
new        CoalOldScr_coal-CCS_mod      70.8           70.8              70.0            70.8           68.7             70.8            70.9
           CoalOldUns_coal-CCS_mod       5.5            5.5               5.3             5.5            5.1              5.5             5.5
           Gas-CC_Gas-CC-CCS_mod         6.6            4.0               NaN             3.8            0.1              1.4             1.3
           Gas-CC_H2-CC                  0.0            0.0               0.0             0.0            0.0              0.0             0.0
           coal-ccs_mod                 85.7           85.5              83.4            85.4           81.4             84.7            84.4
           coal-igcc                     4.8            4.8               4.8             4.8            4.8              4.8             4.8
           coal-new_coal-CCS_mod         6.5            6.5               4.9             6.5            4.9              5.3             4.9
           gas-cc                      384.8          363.8             248.8           371.2          205.6            337.8           317.6
           gas-cc-ccs_mod               16.3           13.8              10.4            14.5           10.8             10.4            10.1
           gas-ct                      167.2          167.0             172.5           167.8          157.9            168.5           171.2
           nuclear                      31.4           31.4              31.4            31.4           31.4             31.4            31.4
           nuclear-smr                   NaN          207.2             766.6           116.7         1258.8            329.3           471.9

**Size beside the module's Δrate.** Δ omitted depreciation = omitted depreciation ÷ retail load, run minus
baseline, 2025 ¢/kWh, beside the national Δrate of the unpatched module and of the patched module (§4).

In [9]:
DEP = pd.concat(DEP, ignore_index=True)
base = DEP[DEP["case"] == BASELINE].set_index(["reading", "t"])["omitted_dep_c2025_per_kWh"]
DEP["delta_omitted_dep_c2025_per_kWh"] = DEP["omitted_dep_c2025_per_kWh"].values - base.reindex(pd.MultiIndex.from_frame(DEP[["reading", "t"]])).values
DEP.to_csv(OUT / "rr_capex_omitted_dep.csv", index=False)
modd = DCOMP.groupby(["case", "t"])["delta_c2025_per_kWh"].sum().rename("patched module Δrate")
def nat_rate_u(case):
    g = pd.read_csv(CASES_U[case] / "retail_rate_components.csv", index_col=0)
    g = g[g["t"] >= 2010].fillna(0.0)
    g["bias_correction"] = g["state"].map(BIAS) * g["retail_load"] * 10 / I_2022
    g = g.groupby("t")[COST + ["retail_load"]].sum()
    return g[COST].sum(axis=1) / g["retail_load"] * TO2025 / 10
ub = nat_rate_u(BASELINE)
modu = pd.concat({c: nat_rate_u(c) - ub for c in PAIRS}, names=["case", "t"]).rename("unpatched module Δrate")
v = DEP[DEP["case"].isin(PAIRS)].pivot_table(index=["case", "t"], columns="reading", values="delta_omitted_dep_c2025_per_kWh")
v = v.join(modu).join(modd)
display(v[v.index.get_level_values("t").isin([2030, 2035, 2040, 2045, 2050])].round(3))

default 20 years  name ignoring case, else 20  unpatched module Δrate  patched module Δrate
case             t                                                                                                
smr100_aj_p50    2030             0.001                        0.001                   0.001                 0.004
                 2035             0.015                        0.010                  -0.004                 0.023
                 2040             0.059                        0.039                  -0.041                 0.061
                 2045             0.113                        0.074                  -0.093                 0.095
                 2050             0.213                        0.141                  -0.155                 0.192
smr100_cop28_p50 2030             0.043                        0.029                  -0.012                 0.077
                 2035             0.249                        0.164                  -0.126                 0.303
                 2040             0.320                        0.211                  -0.214                 0.302
                 2045             0.522                        0.345                  -0.362                 0.446
                 2050             0.662                        0.451                  -0.451                 0.594
smr100_eia_p50   2030             0.000                        0.000                   0.001                 0.001
                 2035             0.000                        0.000                   0.001                 0.001
                 2040             0.104                        0.069                  -0.070                 0.112
                 2045             0.100                        0.066                  -0.084                 0.069
                 2050             0.105                        0.069                  -0.077                 0.065
smr100_eo_p50    2030             0.047                        0.031                  -0.022                 0.073
                 2035             0.315                        0.208                  -0.213                 0.333
                 2040             0.558                        0.369                  -0.433                 0.490
                 2045             0.886                        0.587                  -0.655                 0.740
                 2050             1.145                        0.773                  -0.772                 1.018
smr100_iaea_p50  2030             0.023                        0.015                  -0.004                 0.043
                 2035             0.133                        0.088                  -0.059                 0.170
                 2040             0.198                        0.130                  -0.125                 0.196
                 2045             0.244                        0.160                  -0.184                 0.179
                 2050             0.277                        0.189                  -0.213                 0.210
smr100_mck_p50   2030             0.020                        0.014                  -0.007                 0.035
                 2035             0.133                        0.088                  -0.071                 0.159
                 2040             0.225                        0.148                  -0.153                 0.217
                 2045             0.335                        0.221                  -0.252                 0.269
                 2050             0.425                        0.287                  -0.320                 0.342

**O30, pumped-hydro.** `hydnd` is among the rows that drop out, so O30 reaches the module's output only
through `pumped-hydro`, whose name matches. The module sets its historical cost to `rsc_dat` cost ÷ cap of the
bin with the lowest such ratio in each region (`calculate_historical_capex.py:105-127`), and fills regions
without supply-curve data with the mean of the regional values (lines 136-142). Reading `rsc_dat` cost as
$/MW for the same bin gives the second column. Depreciation over the module's period, 2026 on.

In [10]:
rsc = pd.read_csv(CAPDIR / BASELINE / "o30_rsc_dat.csv")
rsc["i"] = rsc["i"].str.lower()
pv_ = rsc[rsc["i"] == "pumped-hydro"].pivot_table(values="Value", columns="sc_cat", index=["r", "i", "rscbin"]).reset_index().fillna(0.0)
pv_["module_cost"] = pv_["cost"] / pv_["cap"].replace(0, np.nan)
pick = pv_.sort_values("module_cost").drop_duplicates(["r", "i"]).dropna(subset=["module_cost"])
ph = pd.read_csv(CAPDIR / BASELINE / "o30_df_capex_init.csv")
ph = ph[(ph["i"] == "pumped-hydro") & (ph["t"] >= 1960)].merge(pick[["r", "module_cost", "cost"]].rename(columns={"r": "region"}), on="region", how="left")
assert np.allclose(ph["capex"] / ph["cap_new"], ph["module_cost"].fillna(pick["module_cost"].mean()), rtol=1e-6), "module's pumped-hydro cost reproduced"
ph["capex_per_MW"] = ph["cost"].fillna(pick["cost"].mean()) * ph["cap_new"]
E = int(ROWS[BASELINE].query("i == 'pumped-hydro' and src == 'historical'")["eval_period"].iloc[0])
d0 = dep_by_year(ph.assign(eval_period=E), 2026, 2050)
d1 = dep_by_year(ph.drop(columns="capex").rename(columns={"capex_per_MW": "capex"}).assign(eval_period=E), 2026, 2050)
nat0 = pd.read_csv(CASES[BASELINE] / "retail_rate_components.csv", index_col=0).groupby("t")[["retail_load", "cap_gen_dep_expense"]].sum().reindex(range(2026, 2051))
O30T = pd.DataFrame({"t": range(2026, 2051), "module_dep_2004B": d0.values / 1e9, "per_MW_dep_2004B": d1.values / 1e9,
                     "national_cap_gen_dep_2004B": nat0["cap_gen_dep_expense"].values / 1e9,
                     "extra_c2025_per_kWh": ((d1 - d0) / nat0["retail_load"] / 10 * TO2025).values})
O30T.to_csv(OUT / "rr_o30_pumped_hydro.csv", index=False)
print(f"pumped-hydro historical builds 1960-2010: {ph['cap_new'].sum():,.0f} MW; capex 2004 $B: module {ph['capex'].sum()/1e9:.3f}, "
      f"per-MW reading {ph['capex_per_MW'].sum()/1e9:.1f}; evaluation period {E} years")
display(O30T[O30T["t"].isin([2026, 2028, 2030, 2031, 2032])].round(4))
same = {c: pd.read_csv(CAPDIR / c / "o30_df_capex_init.csv").query("i != 'nuclear'").reset_index(drop=True)
        .equals(pd.read_csv(CAPDIR / BASELINE / "o30_df_capex_init.csv").query("i != 'nuclear'").reset_index(drop=True)) for c in CASES}
print("historical rows other than nuclear equal to the baseline's:", same)

pumped-hydro historical builds 1960-2010: 22,914 MW; capex 2004 $B: module 0.039, per-MW reading 42.1; evaluation period 30 years


,t,module_dep_2004B,per_MW_dep_2004B,national_cap_gen_dep_2004B,extra_c2025_per_kWh
0,2026,0.0,0.0096,39.9660,0.0004
2,2028,0.0,0.0091,45.3699,0.0004
4,2030,0.0,0.0087,49.4492,0.0003
5,2031,0.0,0.0085,50.7070,0.0003
6,2032,0.0,0.0000,56.2835,0.0000


historical rows other than nuclear equal to the baseline's: {'fbR_none': True, 'smr100_aj_p50': True, 'smr100_cop28_p50': True, 'smr100_eia_p50': True, 'smr100_eo_p50': True, 'smr100_iaea_p50': True, 'smr100_mck_p50': True}


## 7. What reads these exports

The H3 review page (notebook 25), after Ethan confirms A21, and the observation log. Open items that
bear on these numbers: the 20-year fill for the rows the patched module still finds no period for (§6), O30
(§6), and the years before 2024, where the run and baseline differ in transmission and converter capital
(O31).